这个代码2025年10月重启EPFLUX计算的测试代码，只使用ERA5数据进行计算看计算程序是否正确。

In [ ]:
# ============================================================
# Block A — Per-member daily EP-flux cache + Group aggregates
#   - Stage-1: 对每个成员文件逐一计算“日尺度”EP-flux与DIV（含顶部外推）
#              并同时提取 u(60°N) 在 50/10 hPa 的逐日序列；全部写入 raw 缓存
#   - Stage-2: 基于 raw 缓存：
#              (i) 组内“月平均”（与旧版一致，供 BlockB 使用）
#              (ii) 组内“破裂日前30天平均”（先各自成员算，再对成员做组平均，供 BlockC 使用）
#   - 破裂日判据（60°N）：50 hPa: u<7；10 hPa: u<0；且之后不出现 >10 连续天“高于阈值”的回升
#   - Reference 仅在 0008-02-01 ~ 0008-06-01 窗口内确定破裂日与求平均
#   - 全流程不改变你现有的 monthly 文件/名称；新增 _raw_epflux 与 _preBreakdown*.npz
# ============================================================

import os, glob, json
import numpy as np
import xarray as xr
import pandas as pd
import aostools_functions as ac

# ---------------- 基本路径 ----------------
OUT_ROOT   = "/home/weiji/restart_exam/plots/epflux_speedtest/WACCMnew20251019"
RAW_DIR    = os.path.join(OUT_ROOT, "_raw_epflux")   # 逐成员“日尺度”EP缓存
os.makedirs(OUT_ROOT, exist_ok=True)
os.makedirs(RAW_DIR, exist_ok=True)

# ---------------- 场景定义 ----------------
SCENARIOS = [
    ("0008_Feb_couple",
     "/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/"
     "BWCN.e122.f19_g16.002_0008/Feb/BWCN.e122.f19_g16.002.0008-02.*.nc"),

    ("0008_Mar_couple",
     "/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/"
     "BWCN.e122.f19_g16.002_0008/Mar/BWCN.e122.f19_g16.002.0008-03.*.nc"),

    ("0008_Feb_nocouple",
     "/home/weiji/restart_exam/nocouple_data/0008/Feb_NOCOUPL/0008-02/*.nc"),

    ("0008_Mar_nocouple",
     "/home/weiji/restart_exam/nocouple_data/0008/Mar_NOCOUPL/0008-03/*.nc"),

    # reference：多年份拼接文件，仅取 0008-02~06
    ("0008_long_3.1_6.1",
     "/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/"
     "BWCN.e122.f19_g16.002/BWCN.e122.f19_g16.002.cam.h3.0001-0023.int.nc"),
]

# ---------------- 月份配置 ----------------
def months_for_tag(tag: str):
    if "Feb" in tag:   # Feb 场景：2–5月
        return [2,3,4,5]
    if "Mar" in tag:   # Mar 场景：3–5月
        return [3,4,5]
    if "long" in tag:  # long：2–5月
        return [2,3,4,5]
    return [2,3,4,5]

# ---------------- 工具函数：安全读取 + 时间处理 ----------------
def _open_one_nc(path):
    """
    用 cftime 读取一个 NetCDF 文件，不做时序拼接。
    返回 xr.Dataset（time 为 cftime 对象数组）。
    """
    # 避免 open_mfdataset 的排序&坐标问题，这里逐文件 open_dataset
    time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
    ds = xr.open_dataset(path, decode_times=time_coder, mask_and_scale=True)
    return ds

def _cftime_to_datestr_array(time_da):
    """
    将 cftime.DatetimeNoLeap 等转为 'YYYY-MM-DD' 的 numpy 数组（字符串）。
    只保留到“日”，方便做按月筛选和窗口切片。
    """
    out = np.array([f"{t.year:04d}-{t.month:02d}-{t.day:02d}" for t in time_da.values], dtype="U10")
    return out

def _restrict_reference_0008_Feb_to_Jun(ds, tag):
    """
    对 reference 数据：只保留 0008-02-01 ~ 0008-06-01，
    以避免跨年份混入；其他场景原样返回。
    """
    if "long" not in tag:
        return ds
    # 该文件 time 单位是 from 0001-01-01 开始；仍以 cftime 字符串筛选
    tstr = _cftime_to_datestr_array(ds.time)
    mask = (tstr >= "0008-02-01") & (tstr <= "0008-06-01")
    if mask.sum() == 0:
        return ds
    return ds.isel(time=np.nonzero(mask)[0])

# ---------------- 垂直层前处理：Pa->hPa + 1–100 hPa + 顶部外推 ----------------
EXTRA_LEVELS_HPA = np.array([0.05, 0.04, 0.03, 0.02, 0.01, 0.005, 0.001, 0.0005, 0.0001], dtype=float)

def _prep_vertical_and_extrap(ds):
    """
    将 plev( Pa ) → plev_hpa( hPa )，保留 1–100 hPa，并向上外推到附加层
    返回：新 xr.Dataset（仅保留 U,V,T），coords: time, lat(若有 lon 也保留), plev_hpa
    """
    if "plev" not in ds.dims and "plev" not in ds.coords:
        raise KeyError("Dataset has no 'plev' dimension/coord.")

    ds = ds.sortby("plev", ascending=False).sel(plev=slice(10000.0, 100.0))  # 100..1 hPa

    for var in ("U","V","T"):
        if var not in ds:
            raise KeyError(f"Variable '{var}' missing.")
        ds[var] = ds[var].where(np.isfinite(ds[var]) & (np.abs(ds[var]) < 1e20))

    plev_hpa_vals = (ds["plev"].values / 100.0).astype(float)
    ds = ds.assign_coords(plev_hpa=("plev", plev_hpa_vals)).swap_dims({"plev":"plev_hpa"})
    ds = ds.sortby("plev_hpa").sel(plev_hpa=slice(1.0, 100.0))

    existing = ds["plev_hpa"].values.astype(float)
    target_levels = np.sort(np.unique(np.concatenate([existing, EXTRA_LEVELS_HPA])))

    interp = {}
    for var in ("U","V","T"):
        interp[var] = ds[var].interp(plev_hpa=target_levels, kwargs={"fill_value":"extrapolate"}, assume_sorted=True)

    coords = {"time": ds.time, "lat": ds.lat, "plev_hpa": ("plev_hpa", target_levels)}
    if "lon" in ds.coords: coords["lon"] = ds.lon

    out = xr.Dataset(
        data_vars={v: (interp[v].dims, interp[v].data) for v in ("U","V","T")},
        coords=coords
    )
    return out

# ---------------- 取 60°N 与两层风速 ----------------
def _nearest_lat_index(lat_vals, target=60.0):
    return int(np.argmin(np.abs(np.asarray(lat_vals, dtype=float) - float(target))))

def _extract_u60_timeseries(ds_hpa, hpa, lat_idx):
    """
    从 ds_hpa 中取某一层( hpa )、某一纬线( lat_idx )的 u(经向平均)逐日序列。
    返回 u_series (time, ) — m/s
    """
    # 最近邻插值到指定 hPa 层（保证层存在）
    target = float(hpa)
    p = ds_hpa.plev_hpa.values.astype(float)
    # 找最近层索引
    k = int(np.argmin(np.abs(p - target)))
    u = ds_hpa["U"][:, k, lat_idx, :]
    # 经向平均（zonal mean on that latitude line）
    u_zm = np.nanmean(u.values, axis=-1)  # (time,)
    return u_zm

# ---------------- 破裂日搜索（含“>10 连续天回升”剔除） ----------------
def _find_breakdown_date(time_str, u_series, thr, below=True):
    """
    输入：
      time_str: ndarray[str], 'YYYY-MM-DD'
      u_series: ndarray[float], 与 time 对齐
      thr: 阈值（7 或 0）
      below: True 表示 “u<thr”为破裂条件；False 表示 “u>thr”
    规则：
      找到第一个满足条件的日 i，使得在 i 之后不出现“>10 连续天”的反向回升。
      若出现，则跳过该候选，继续向后找。
    返回：索引 i（int）或 None
    """
    cond = (u_series < thr) if below else (u_series > thr)
    n = cond.size
    # 预生成“反向回升”的布尔序列
    rebound = (u_series >= thr) if below else (u_series <= thr)

    i = 0
    while i < n:
        if cond[i]:
            # 检查后续是否出现 >10 连续天 rebound
            # 计算 i+1.. 的 run-length
            if i+1 < n:
                r = rebound[i+1:]
                # 连续计数
                max_run = 0
                cur = 0
                for x in r:
                    if x: 
                        cur += 1
                        max_run = max(max_run, cur)
                    else:
                        cur = 0
                if max_run > 10:
                    # 此候选无效，继续寻找下一个 cond True 的位置
                    i += 1
                    continue
            return i
        i += 1
    return None

# ---------------- Stage-1：逐成员文件计算并缓存“日尺度”EP ----------------
def stage1_cache_raw_ep(tag, pattern):
    """
    对某组（tag）下的每个成员文件逐一：
      1) 打开 → 限定参考期（仅 reference） → 垂直前处理与顶部外推
      2) 计算日尺度 EP-flux & DIV（ac.ComputeEPfluxDiv）
      3) 提取 u(60°N) at 50/10 hPa 序列
      4) 写入 RAW_DIR/tag/xxx.npz：包含 lat, plev_hpa, time_str, ep1, ep2, div, u50_60N, u10_60N
    """
    files = sorted(glob.glob(pattern))
    if len(files) == 0 and "long" not in tag:
        print(f"  ⚠️ No files for {tag}: {pattern}")
        return

    os.makedirs(os.path.join(RAW_DIR, tag), exist_ok=True)
    print(f"\n=== Stage-1 raw EP: {tag} ===")

    # reference 是单文件
    if "long" in tag:
        ds0 = _open_one_nc(pattern)
        ds0 = _restrict_reference_0008_Feb_to_Jun(ds0, tag)
        ds = _prep_vertical_and_extrap(ds0)

        lat = ds["lat"].values
        plev = ds["plev_hpa"].values.astype(float)
        tstr = _cftime_to_datestr_array(ds["time"])
        lat_idx = _nearest_lat_index(lat, 60.0)

        # Compute EP（time,p,lat,lon → 返回 ep1,ep2,div1,div2）
        ep1, ep2, d1, d2 = ac.ComputeEPfluxDiv(
            lat=lat, pres=plev, u=ds["U"].values, v=ds["V"].values, t=ds["T"].values,
            wave=-1, do_ubar=True
        )
        div = (d1 + d2).astype(np.float64)

        # 60N u-series
        u50 = _extract_u60_timeseries(ds, 50.0, lat_idx)
        u10 = _extract_u60_timeseries(ds, 10.0, lat_idx)

        out_path = os.path.join(RAW_DIR, tag, f"{tag}__{os.path.basename(pattern).replace('.nc','')}.npz")
        np.savez_compressed(out_path,
            lat=lat, plev=plev, time_str=tstr,
            ep1=ep1, ep2=ep2, div=div,
            u50_60N=u50, u10_60N=u10
        )
        print(f"  📦 raw cached: {out_path}")
        return

    # hindcast：多成员多文件
    for fp in files:
        ds0 = _open_one_nc(fp)
        ds = _prep_vertical_and_extrap(ds0)

        lat = ds["lat"].values
        plev = ds["plev_hpa"].values.astype(float)
        tstr = _cftime_to_datestr_array(ds["time"])
        lat_idx = _nearest_lat_index(lat, 60.0)

        ep1, ep2, d1, d2 = ac.ComputeEPfluxDiv(
            lat=lat, pres=plev, u=ds["U"].values, v=ds["V"].values, t=ds["T"].values,
            wave=-1, do_ubar=True
        )
        div = (d1 + d2).astype(np.float64)

        u50 = _extract_u60_timeseries(ds, 50.0, lat_idx)
        u10 = _extract_u60_timeseries(ds, 10.0, lat_idx)

        tag_dir = os.path.join(RAW_DIR, tag)
        os.makedirs(tag_dir, exist_ok=True)
        out_path = os.path.join(tag_dir, f"{tag}__{os.path.basename(fp).replace('.nc','')}.npz")
        np.savez_compressed(out_path,
            lat=lat, plev=plev, time_str=tstr,
            ep1=ep1, ep2=ep2, div=div,
            u50_60N=u50, u10_60N=u10
        )
        print(f"  📦 raw cached: {out_path}")

# ---------------- Stage-2：基于 raw 的组内汇总 ----------------
def _month_mask(time_str, month_int):
    # time_str: 'YYYY-MM-DD'
    return np.array([int(s[5:7]) == month_int for s in time_str], dtype=bool)

def _window_last_30_idx(idx_break):
    # 破裂日(含)向前 29 天 → 共 30 天。若不足 30，从 0 开始。
    start = max(0, idx_break - 29)
    stop  = idx_break + 1  # inclusive
    return np.arange(start, stop, dtype=int)

def _aggregate_group_from_raw(tag):
    """
    从 RAW_DIR/tag 读取全部成员：
      1) 组内“月平均”：对每个成员先对当月做日平均，再 across 成员做平均；
         输出 epflux_mean_allwaves_{tag}_mXX_extrap.npz & div 对应文件
      2) 组内“破裂日前30天平均”：先在成员内找破裂日→取窗口→平均；再 across 成员做平均；
         输出 epflux_mean_allwaves_{tag}_preBreakdown(50/10)hPa_extrap.npz & div 对应文件
    """
    print(f"\n=== Stage-2 aggregate: {tag} ===")
    raw_files = sorted(glob.glob(os.path.join(RAW_DIR, tag, "*.npz")))
    if len(raw_files) == 0:
        print(f"  ⚠️ no raw caches for {tag}")
        return

    # 读第一个文件获得网格
    head = np.load(raw_files[0])
    lat = head["lat"]; plev = head["plev"]
    # 月平均
    for m in months_for_tag(tag):
        ep1_list, ep2_list, dv_list = [], [], []
        for rf in raw_files:
            R = np.load(rf)
            tmask = _month_mask(R["time_str"], m)
            if tmask.any():
                ep1_list.append(np.nanmean(R["ep1"][tmask, ...], axis=0))
                ep2_list.append(np.nanmean(R["ep2"][tmask, ...], axis=0))
                dv_list.append(np.nanmean(R["div"][tmask, ...], axis=0))
        if len(ep1_list) == 0:
            continue
        ep1m = np.nanmean(np.stack(ep1_list, axis=0), axis=0)
        ep2m = np.nanmean(np.stack(ep2_list, axis=0), axis=0)
        dvm  = np.nanmean(np.stack(dv_list , axis=0), axis=0)

        f_mean = os.path.join(OUT_ROOT, f"epflux_mean_allwaves_{tag}_m{m:02d}_extrap.npz")
        f_div  = os.path.join(OUT_ROOT, f"epflux_div_mean_allwaves_{tag}_m{m:02d}_extrap.npz")
        np.savez_compressed(f_mean, lat=lat, plev=plev, ep1=ep1m, ep2=ep2m)
        np.savez_compressed(f_div , lat=lat, plev=plev, div=dvm)
        print(f"  ✅ monthly cache: {tag} m{m:02d}")

    # 破裂日前30天平均（50hPa/10hPa 各一组）
    def _do_prebreak(thr, below, label):
        ep1_mem, ep2_mem, dv_mem = [], [], []
        for rf in raw_files:
            R = np.load(rf)
            tstr = R["time_str"]
            # 选择哪条 u 序列
            u_series = R["u50_60N"] if label == "50hPa" else R["u10_60N"]
            idx = _find_breakdown_date(tstr, u_series, thr=thr, below=below)
            if idx is None:
                continue
            idxs = _window_last_30_idx(idx)
            ep1_mem.append(np.nanmean(R["ep1"][idxs, ...], axis=0))
            ep2_mem.append(np.nanmean(R["ep2"][idxs, ...], axis=0))
            dv_mem .append(np.nanmean(R["div"][idxs, ...], axis=0))
        if len(ep1_mem) == 0:
            print(f"  ⚠️ no valid breakdown for {tag} {label}")
            return
        ep1g = np.nanmean(np.stack(ep1_mem, axis=0), axis=0)
        ep2g = np.nanmean(np.stack(ep2_mem, axis=0), axis=0)
        dvg  = np.nanmean(np.stack(dv_mem , axis=0), axis=0)

        f_mean = os.path.join(OUT_ROOT, f"epflux_mean_allwaves_{tag}_preBreakdown{label}_extrap.npz")
        f_div  = os.path.join(OUT_ROOT, f"epflux_div_mean_allwaves_{tag}_preBreakdown{label}_extrap.npz")
        np.savez_compressed(f_mean, lat=lat, plev=plev, ep1=ep1g, ep2=ep2g)
        np.savez_compressed(f_div , lat=lat, plev=plev, div=dvg)
        print(f"  ✅ 30d pre-breakdown: {tag} preBreakdown{label}  {tstr[max(0,idx-29)]} → {tstr[idx]}")

    # 50 hPa：u < 7
    _do_prebreak(thr=7.0, below=True,  label="50hPa")
    # 10 hPa：u < 0
    _do_prebreak(thr=0.0, below=True,  label="10hPa")

# ---------------- 主流程 ----------------
if __name__ == "__main__":
    # Stage-1：逐成员原始 EP（天尺度）缓存
    for tag, pattern in SCENARIOS:
        stage1_cache_raw_ep(tag, pattern)

    # Stage-2：组内聚合（仍保留你原有的“月平均”产物 + 新增两套 preBreakdown）
    for tag, _ in SCENARIOS:
        _aggregate_group_from_raw(tag)

    print("\nAll caches generated from per-file raw EP: monthly and 30-day pre-breakdown (50/10 hPa).")


In [ ]:
# ============================================================
# Block B — Plot monthly EP-flux (vectors over divergence)
#   - step_lat = 2 (denser arrows)
#   - Scientific title with initialization-month info
#   - Explicit ozone-feedback description (ON/OFF)
#   - Month labels: Feb, Mar, Apr, May
#   - Display strictly 1–100 hPa
#   - Compatible with new "_extrap.npz" files
# ============================================================

import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import aostools_functions as ac

root_dir = "/home/weiji/restart_exam/plots/epflux_speedtest/WACCMnew20251019"
scenarios = [
    "0008_Feb_couple",
    "0008_Mar_couple",
    "0008_Feb_nocouple",
    "0008_Mar_nocouple",
    "0008_long_3.1_6.1",
]

# ---- month & experiment meta ----
def months_for_tag(tag: str):
    if "Feb" in tag:
        return [2,3,4,5]
    if "Mar" in tag:
        return [3,4,5]
    if "long" in tag:
        return [2,3,4,5]
    return [2,3,4,5]

MONTH_TXT = {2:"Feb", 3:"Mar", 4:"Apr", 5:"May"}

def experiment_title(tag: str) -> str:
    toks = [t.lower() for t in tag.split("_")]  # e.g. ['0008','feb','nocouple']
    if "long" in toks or "long" in tag.lower():
        return "Long-run reference simulation"
    if "nocouple" in toks:
        return "Hindcast (radiative ozone feedback OFF)"
    if "couple" in toks:
        return "Hindcast (radiative ozone feedback ON)"
    return "Experiment"

def init_month_subtitle(tag: str) -> str:
    if "Feb" in tag:
        return "Initialized in February (0008)"
    if "Mar" in tag:
        return "Initialized in March (0008)"
    if "long" in tag:
        return "Multi-year integration used as reference"
    return ""

# ---- plot settings ----
levels_div  = np.linspace(-4, 4, 17)
step_lat    = 2
step_plev   = 1
arrow_scale = 2e16

for tag in scenarios:
    exp_title = experiment_title(tag)
    sub_title = init_month_subtitle(tag)

    for m in months_for_tag(tag):
        month_txt = MONTH_TXT[m]

        # --- 优先读取 "_extrap" 文件版本 ---
        mean_file = os.path.join(root_dir, f"epflux_mean_allwaves_{tag}_m{m:02d}_extrap.npz")
        div_file  = os.path.join(root_dir, f"epflux_div_mean_allwaves_{tag}_m{m:02d}_extrap.npz")
        if not (os.path.exists(mean_file) and os.path.exists(div_file)):
            # fallback to old names
            mean_file = mean_file.replace("_extrap", "")
            div_file  = div_file.replace("_extrap", "")
            if not (os.path.exists(mean_file) and os.path.exists(div_file)):
                print(f"⚠️ Missing cache: {tag} {month_txt}")
                continue

        V = np.load(mean_file)
        D = np.load(div_file)

        lat  = V["lat"]
        plev = V["plev"].astype(float)  # hPa
        F1   = V["ep1"]
        F2   = V["ep2"]
        DIV  = D["div"]

        # --- 严格裁剪 1–100 hPa ---
        sel = (plev >= 1.0) & (plev <= 100.0)
        plev = plev[sel]
        F1, F2, DIV = F1[sel, :], F2[sel, :], DIV[sel, :]

        # ---- plotting ----
        fig, ax = plt.subplots(figsize=(8, 6))

        cf = ax.contourf(lat, plev, DIV, levels=levels_div, cmap="RdBu_r", extend="both")

        f1 = xr.DataArray(F1, coords={"plev": plev, "lat": lat}, dims=["plev","lat"]).transpose("lat","plev")
        f2 = xr.DataArray(F2, coords={"plev": plev, "lat": lat}, dims=["plev","lat"]).transpose("lat","plev")
        f1s = f1.isel(lat=slice(None,None,step_lat), plev=slice(None,None,step_plev))
        f2s = f2.isel(lat=slice(None,None,step_lat), plev=slice(None,None,step_plev))

        ac.PlotEPfluxArrows(
            f1s.lat, f1s.plev, f1s, f2s,
            fig, ax,
            xlim=[-90, 90],
            ylim=[100, 2],
            yscale="log",
            scale=arrow_scale
        )

        ax.set_xlim([-90, 90])
        ax.set_ylim([100, 2])
        ax.set_yscale("log")
        ax.set_yticks([100, 60, 30, 10, 5, 2])
        ax.set_yticklabels(["100", "60", "30", "10", "5", "2"])
        ax.set_xlabel("Latitude (°)")
        ax.set_ylabel("Pressure (hPa)")

        # --- scientific title ---
        ax.set_title(
            f"{exp_title} — {sub_title}\n"
            f"{month_txt} 0008 | EP-flux vectors over divergence "
            f"(all waves, do_ubar = True)",
            fontsize=10
        )

        cbar = fig.colorbar(cf, ax=ax, pad=0.02)
        cbar.set_label("EP-flux divergence [m s⁻¹ day⁻¹]")

        outfile = os.path.join(root_dir, f"epflux_combined_allwaves_{tag}_{month_txt}_extrap.png")
        fig.savefig(outfile, dpi=300, bbox_inches="tight")
        plt.close(fig)
        print(f"✅ Saved: {outfile}")

print("\nAll EP-flux plots generated (strict 1–100 hPa range, using _extrap files).")


In [ ]:
# ============================================================
# Block C — Plot pre-breakdown (30-day) group-mean EP-flux
#   - Two criteria: 50 hPa (< 7 m s⁻¹), 10 hPa (< 0 m s⁻¹)
#   - Each criterion plotted separately per scenario (tag)
#   - Strict 1–100 hPa; dense but readable arrows; NG-style
# ============================================================

import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import aostools_functions as ac

root_dir = OUT_ROOT  # 与 Block A 一致
scenarios = [t for t, _ in SCENARIOS]

levels_div  = np.linspace(-4, 4, 17)
step_lat    = 2
step_plev   = 1
arrow_scale = 2e16

def _try_load(tag: str, label: str):
    f_mean = os.path.join(root_dir, f"epflux_mean_allwaves_{tag}_{label}_extrap.npz")
    f_div  = os.path.join(root_dir, f"epflux_div_mean_allwaves_{tag}_{label}_extrap.npz")
    if not (os.path.exists(f_mean) and os.path.exists(f_div)):
        return None, None
    V = np.load(f_mean); D = np.load(f_div)
    return V, D

def _exp_title(tag: str) -> str:
    toks = [t.lower() for t in tag.split("_")]
    if "long" in toks or "long" in tag.lower():
        return "Long-run reference (0008 pre-breakdown window)"
    if "nocouple" in toks:
        return "Hindcast (radiative ozone feedback OFF)"
    if "couple" in toks:
        return "Hindcast (radiative ozone feedback ON)"
    return "Experiment"

def _init_subtitle(tag: str) -> str:
    if "Feb" in tag:
        return "Initialized in February (0008)"
    if "Mar" in tag:
        return "Initialized in March (0008)"
    if "long" in tag:
        return "Group-mean of per-experiment 30-day windows (0008 only)"
    return ""

def _plot_one(tag: str, label: str, pretty: str):
    V, D = _try_load(tag, label)
    if V is None:
        print(f"⚠️ Missing preBD cache: {tag} {label}")
        return

    lat  = V["lat"]
    plev = V["plev"].astype(float)  # hPa
    F1   = V["ep1"]
    F2   = V["ep2"]
    DIV  = D["div"]

    # 裁剪 1–100 hPa
    sel = (plev >= 1.0) & (plev <= 100.0)
    plev = plev[sel]
    F1, F2, DIV = F1[sel, :], F2[sel, :], DIV[sel, :]

    fig, ax = plt.subplots(figsize=(8, 6))
    cf = ax.contourf(lat, plev, DIV, levels=levels_div, cmap="RdBu_r", extend="both")

    f1 = xr.DataArray(F1, coords={"plev": plev, "lat": lat}, dims=["plev","lat"]).transpose("lat","plev")
    f2 = xr.DataArray(F2, coords={"plev": plev, "lat": lat}, dims=["plev","lat"]).transpose("lat","plev")

    f1s = f1.isel(lat=slice(None,None,step_lat), plev=slice(None,None,step_plev))
    f2s = f2.isel(lat=slice(None,None,step_lat), plev=slice(None,None,step_plev))

    ac.PlotEPfluxArrows(
        f1s.lat, f1s.plev, f1s, f2s,
        fig, ax,
        xlim=[-90, 90],
        ylim=[100, 2],
        yscale="log",
        scale=arrow_scale
    )

    ax.set_xlim([-90, 90])
    ax.set_ylim([100, 2])
    ax.set_yscale("log")
    ax.set_yticks([100, 60, 30, 10, 5, 2])
    ax.set_yticklabels(["100", "60", "30", "10", "5", "2"])
    ax.set_xlabel("Latitude (°)")
    ax.set_ylabel("Pressure (hPa)")

    ax.set_title(
        f"{_exp_title(tag)} — {_init_subtitle(tag)}\n"
        f"30-day pre-breakdown mean | Criterion: {pretty} | "
        f"EP-flux vectors over divergence (all waves, do_ubar=True)",
        fontsize=10
    )

    cbar = fig.colorbar(cf, ax=ax, pad=0.02)
    cbar.set_label("EP-flux divergence [m s⁻¹ day⁻¹]")

    outfile = os.path.join(root_dir, f"epflux_{label}_{tag}_extrap.png")
    fig.savefig(outfile, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"✅ Saved: {outfile}")

# 主循环：每个 tag 各画两张（50hPa 与 10hPa）
for tag in scenarios:
    _plot_one(tag, "preBreakdown50hPa", "50 hPa: u(60°N) < 7 m s⁻¹")
    _plot_one(tag, "preBreakdown10hPa", "10 hPa: u(60°N) < 0 m s⁻¹")

print("\nAll pre-breakdown (30d) group-mean EP-flux plots generated.")
